# 20_production_serving_artifacts.ipynb — Artefactos de producción para API y cliente

Este notebook prepara la fase de **Puesta en Producción** de DeepWave Canarias.

No entrena modelos. Convierte los resultados de modelado en archivos ligeros y estables para servir desde una API y una aplicación cliente.

## Objetivo

Crear una carpeta `app_data/` con datos listos para:

```text
FastAPI backend
dashboard web
Swagger /docs
demo local
despliegue web
```

## Entradas principales

```text
gold/model_results/multitarget_physical/predictions_val_test_core_targets.parquet
gold/model_results/final_report_multitarget/
gold/multitarget_training_dataset/
```

## Salidas principales

```text
app_data/
├── zones.json
├── forecast_by_zone.json
├── predictions_flat.json
├── latest_predictions.json
├── model_summary.json
├── api_contract.json
├── risk_legend.json
├── surf_legend.json
├── demo_examples.json
├── production_manifest.json
└── production_readiness_report.md
```

## Enfoque de producción

Para la primera versión de producción se usa una estrategia robusta:

```text
predicciones precalculadas
→ JSON ligero
→ API rápida
→ cliente web estable
```

Esto evita que la API tenga que cargar datasets Parquet enormes o 100 modelos `.pkl` al arrancar.

## Celda 0 — Entorno local / Colab

In [1]:
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Entorno local detectado. No se monta Google Drive.")

Entorno local detectado. No se monta Google Drive.


## Celda 1 — Imports, rutas y configuración

In [2]:
from pathlib import Path
import os
import sys
import json
import shutil
import platform
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import pyarrow.dataset as ds
except Exception:
    ds = None

# ---------------------------------------------------------------------
# RUTAS — MAC FIRST
# ---------------------------------------------------------------------
RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
else:
    # Notebook esperado en deep-wave-canarias/notebooks/
    BASE_DIR = Path.cwd().parent.resolve()

GOLD_DIR = BASE_DIR / "gold"
MODEL_RESULTS_DIR = GOLD_DIR / "model_results"

PHYSICAL_RESULTS_DIR = MODEL_RESULTS_DIR / "multitarget_physical"
RISK_RESULTS_DIR = MODEL_RESULTS_DIR / "risk_modules"
SURF_RESULTS_DIR = MODEL_RESULTS_DIR / "surf_score"
FINAL_RESULTS_DIR = MODEL_RESULTS_DIR / "final_report_multitarget"

INPUT_GOLD_MULTITARGET_DIR = GOLD_DIR / "multitarget_training_dataset"

APP_DATA_DIR = BASE_DIR / "app_data"
APP_DATA_ARCHIVE_DIR = APP_DATA_DIR / "_archive"
REPORTS_DIR = BASE_DIR / "reports"
DOCS_DIR = BASE_DIR / "docs"

for d in [APP_DATA_DIR, APP_DATA_ARCHIVE_DIR, REPORTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHYSICAL_PREDICTIONS_PATH = PHYSICAL_RESULTS_DIR / "predictions_val_test_core_targets.parquet"

# ---------------------------------------------------------------------
# CONFIGURACIÓN DE PRODUCCIÓN
# ---------------------------------------------------------------------
HORIZONS_HOURS = [3, 6, 12, 24, 48]

# Split preferido para demo. test = periodo no visto.
PREFERRED_SPLIT_ORDER = ["test", "val", "train"]

# Targets físicos que intentaremos incluir en API.
TARGET_PRIORITY = [
    "hs",
    "tm02",
    "tp",
    "swell_height",
    "swell_period",
    "wave_direction",
    "swell_direction",
    "wind_speed",
    "wind_direction",
    "sea_level",
    "daily_tidal_range",
    "wind_wave_height",
]

# Número máximo de zonas a incluir. None = todas.
MAX_ZONES = None

# Crear backup de app_data/*.json anteriores antes de sobrescribir.
BACKUP_EXISTING_APP_DATA = True

# Redondeo para JSON.
ROUND_DECIMALS = 4

# Umbrales operativos coherentes con notebooks 16/17.
RISK_THRESHOLDS = {
    "general": {
        "moderate_hs": 1.0,
        "high_hs": 2.0,
        "extreme_hs": 3.0,
    },
    "beach": {
        "moderate_hs": 1.0,
        "high_hs": 1.8,
        "extreme_hs": 2.7,
        "high_period": 12.0,
        "strong_wind": 10.0,
    },
    "navigation": {
        "moderate_hs": 1.2,
        "high_hs": 2.0,
        "extreme_hs": 3.0,
        "strong_wind": 10.0,
        "very_strong_wind": 14.0,
        "long_period": 11.0,
    },
}

SURF_SCORE_CONFIG = {
    "ideal_hs_min": 0.8,
    "ideal_hs_max": 2.5,
    "ideal_period_min": 9.0,
    "ideal_period_good": 12.0,
    "bad_wind_speed": 10.0,
    "very_bad_wind_speed": 14.0,
}

RISK_LABELS = {
    0: {"label": "low", "label_es": "bajo", "color": "#2ecc71"},
    1: {"label": "moderate", "label_es": "medio", "color": "#f1c40f"},
    2: {"label": "high", "label_es": "alto", "color": "#e67e22"},
    3: {"label": "extreme", "label_es": "extremo", "color": "#e74c3c"},
}

SURF_LABELS = {
    "poor": {"label_es": "malo", "color": "#95a5a6", "min": 0, "max": 2},
    "fair": {"label_es": "regular", "color": "#f1c40f", "min": 2, "max": 4},
    "good": {"label_es": "bueno", "color": "#2ecc71", "min": 4, "max": 6},
    "very_good": {"label_es": "muy bueno", "color": "#3498db", "min": 6, "max": 8},
    "epic": {"label_es": "excelente", "color": "#9b59b6", "min": 8, "max": 10},
}

print("Sistema:", platform.platform())
print("BASE_DIR:", BASE_DIR)
print("APP_DATA_DIR:", APP_DATA_DIR)
print("PHYSICAL_PREDICTIONS_PATH:", PHYSICAL_PREDICTIONS_PATH, "existe:", PHYSICAL_PREDICTIONS_PATH.exists())
print("INPUT_GOLD_MULTITARGET_DIR:", INPUT_GOLD_MULTITARGET_DIR, "existe:", INPUT_GOLD_MULTITARGET_DIR.exists())
print("FINAL_RESULTS_DIR:", FINAL_RESULTS_DIR, "existe:", FINAL_RESULTS_DIR.exists())

Sistema: macOS-26.3.1-arm64-arm-64bit
BASE_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias
APP_DATA_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data
PHYSICAL_PREDICTIONS_PATH: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/multitarget_physical/predictions_val_test_core_targets.parquet existe: True
INPUT_GOLD_MULTITARGET_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/multitarget_training_dataset existe: True
FINAL_RESULTS_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget existe: True


## Celda 2 — Funciones auxiliares de IO y JSON

In [3]:
written_files = []

def json_default(obj):
    """Serializador seguro para numpy/pandas/timestamps."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp,)):
        return obj.isoformat()
    if isinstance(obj, (datetime,)):
        return obj.isoformat()
    if pd.isna(obj):
        return None
    return str(obj)


def safe_float(x, decimals=ROUND_DECIMALS):
    try:
        if x is None:
            return None
        if pd.isna(x):
            return None
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return None
        return round(x, decimals)
    except Exception:
        return None


def safe_int(x):
    try:
        if x is None or pd.isna(x):
            return None
        return int(x)
    except Exception:
        return None


def safe_str(x, default=None):
    try:
        if x is None or pd.isna(x):
            return default
        return str(x)
    except Exception:
        return default


def backup_output(path):
    path = Path(path)
    if path.exists() and BACKUP_EXISTING_APP_DATA:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        dst = APP_DATA_ARCHIVE_DIR / f"{path.name}.bak_{stamp}"
        shutil.copy2(path, dst)
        return dst
    return None


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    backup_output(path)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=json_default)

    written_files.append(str(path))
    print("JSON escrito:", path)
    return path


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    backup_output(path)
    path.write_text(text.strip() + "\n", encoding="utf-8")
    written_files.append(str(path))
    print("Texto escrito:", path)
    return path


def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("AVISO leyendo JSON:", path, e)
        return {}


def read_csv_safe(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("AVISO leyendo CSV:", path, e)
        return pd.DataFrame()


def normalize_timestamp(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def memory_report(name, df):
    mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{name}: shape={df.shape}, memoria≈{mb:.1f} MB")


def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def compact_record(record):
    """Limpia un dict para JSON: numpy -> python, NaN -> None."""
    out = {}
    for k, v in record.items():
        if isinstance(v, (np.integer,)):
            out[k] = int(v)
        elif isinstance(v, (np.floating, float)):
            out[k] = safe_float(v)
        elif isinstance(v, (pd.Timestamp, datetime)):
            out[k] = v.isoformat()
        elif pd.isna(v) if not isinstance(v, (list, dict, tuple)) else False:
            out[k] = None
        else:
            out[k] = v
    return out

## Celda 3 — Funciones de riesgo, surf y recomendaciones

In [4]:
def risk_payload(level):
    level = safe_int(level)
    if level is None:
        return {
            "level": None,
            "label": "unknown",
            "label_es": "desconocido",
            "color": "#7f8c8d",
        }

    data = RISK_LABELS.get(level, RISK_LABELS[3]).copy()
    data["level"] = level
    return data


def assign_risk_general(hs):
    hs = safe_float(hs)
    if hs is None:
        return None
    if hs < RISK_THRESHOLDS["general"]["moderate_hs"]:
        return 0
    if hs < RISK_THRESHOLDS["general"]["high_hs"]:
        return 1
    if hs < RISK_THRESHOLDS["general"]["extreme_hs"]:
        return 2
    return 3


def assign_risk_beach(hs, period=None, wind_speed=None, exposure=None):
    hs = safe_float(hs)
    period = safe_float(period)
    wind_speed = safe_float(wind_speed)
    exposure = safe_float(exposure)

    if hs is None:
        return None

    if hs < RISK_THRESHOLDS["beach"]["moderate_hs"]:
        risk = 0
    elif hs < RISK_THRESHOLDS["beach"]["high_hs"]:
        risk = 1
    elif hs < RISK_THRESHOLDS["beach"]["extreme_hs"]:
        risk = 2
    else:
        risk = 3

    boost = 0

    if period is not None and period >= RISK_THRESHOLDS["beach"]["high_period"] and hs >= 1.2:
        boost += 1

    if wind_speed is not None and wind_speed >= RISK_THRESHOLDS["beach"]["strong_wind"]:
        boost += 1

    if exposure is not None and exposure >= 1 and hs >= 1.0:
        boost += 1

    if boost >= 2:
        risk += 1

    return int(max(0, min(3, risk)))


def assign_risk_navigation(hs, period=None, wind_speed=None):
    hs = safe_float(hs)
    period = safe_float(period)
    wind_speed = safe_float(wind_speed)

    if hs is None:
        return None

    if hs < RISK_THRESHOLDS["navigation"]["moderate_hs"]:
        risk = 0
    elif hs < RISK_THRESHOLDS["navigation"]["high_hs"]:
        risk = 1
    elif hs < RISK_THRESHOLDS["navigation"]["extreme_hs"]:
        risk = 2
    else:
        risk = 3

    if wind_speed is not None and wind_speed >= RISK_THRESHOLDS["navigation"]["strong_wind"]:
        risk += 1

    if wind_speed is not None and wind_speed >= RISK_THRESHOLDS["navigation"]["very_strong_wind"]:
        risk += 1

    if period is not None and period >= RISK_THRESHOLDS["navigation"]["long_period"] and hs >= 1.5:
        risk += 1

    return int(max(0, min(3, risk)))


def angular_diff_deg(a, b):
    a = safe_float(a)
    b = safe_float(b)
    if a is None or b is None:
        return None
    return abs((a - b + 180) % 360 - 180)


def surf_score_from_conditions(hs, period=None, wind_speed=None, wave_direction=None, wind_direction=None, coast_orientation_deg=None):
    hs = safe_float(hs)
    period = safe_float(period)
    wind_speed = safe_float(wind_speed)
    wave_direction = safe_float(wave_direction)
    wind_direction = safe_float(wind_direction)
    coast_orientation_deg = safe_float(coast_orientation_deg)

    if hs is None:
        return None

    score = 0.0

    # Altura.
    if SURF_SCORE_CONFIG["ideal_hs_min"] <= hs <= SURF_SCORE_CONFIG["ideal_hs_max"]:
        score += 3.0
    if 0.5 <= hs <= 3.0:
        score += 1.0
    if hs < 0.4:
        score -= 2.0
    if hs > 3.5:
        score -= 1.5

    # Periodo.
    if period is not None:
        if period >= SURF_SCORE_CONFIG["ideal_period_min"]:
            score += 2.0
        if period >= SURF_SCORE_CONFIG["ideal_period_good"]:
            score += 1.0

    # Viento.
    if wind_speed is not None:
        if wind_speed <= 5.0:
            score += 1.0
        if wind_speed >= SURF_SCORE_CONFIG["bad_wind_speed"]:
            score -= 1.0
        if wind_speed >= SURF_SCORE_CONFIG["very_bad_wind_speed"]:
            score -= 1.0

    # Orientación simplificada.
    orientation_scores = []

    if wave_direction is not None and coast_orientation_deg is not None:
        diff = angular_diff_deg(wave_direction, coast_orientation_deg)
        if diff is not None:
            wave_alignment = (np.cos(np.deg2rad(diff)) + 1) / 2
            orientation_scores.append(max(0, min(1, wave_alignment)))

    if wind_direction is not None and coast_orientation_deg is not None:
        diff = angular_diff_deg(wind_direction, coast_orientation_deg)
        if diff is not None:
            # Offshore simplificado: viento opuesto a orientación costera.
            wind_offshore = max(0, min(1, diff / 180.0))
            orientation_scores.append(wind_offshore)

    if orientation_scores:
        score += float(np.mean(orientation_scores)) * 2.0
    else:
        score += 1.0

    return round(max(0, min(10, score)), 2)


def surf_quality_label(score):
    score = safe_float(score)
    if score is None:
        return None
    if score < 2:
        return "poor"
    if score < 4:
        return "fair"
    if score < 6:
        return "good"
    if score < 8:
        return "very_good"
    return "epic"


def surf_payload(score):
    score = safe_float(score)
    quality = surf_quality_label(score)

    if quality is None:
        return {
            "score": None,
            "quality": "unknown",
            "quality_es": "desconocido",
            "color": "#7f8c8d",
        }

    legend = SURF_LABELS[quality]

    return {
        "score": score,
        "quality": quality,
        "quality_es": legend["label_es"],
        "color": legend["color"],
    }


def recommendation_text(record):
    """Mensaje textual simple para API/dashboard."""
    hs = record.get("hs")
    period = record.get("period")
    wind_speed = record.get("wind_speed")

    beach_level = record.get("risk_beach", {}).get("level")
    navigation_level = record.get("risk_navigation", {}).get("level")
    surf_quality = record.get("surf", {}).get("quality")

    parts = []

    if beach_level is not None:
        if beach_level >= 3:
            parts.append("Riesgo extremo para baño. Evitar zonas expuestas y consultar avisos oficiales.")
        elif beach_level == 2:
            parts.append("Riesgo alto para playa. Precaución especial con oleaje y corrientes.")
        elif beach_level == 1:
            parts.append("Riesgo medio para playa. Condiciones moderadas; mantener precaución.")
        else:
            parts.append("Riesgo bajo para playa en las condiciones previstas.")

    if navigation_level is not None:
        if navigation_level >= 3:
            parts.append("Navegación ligera no recomendable.")
        elif navigation_level == 2:
            parts.append("Navegación ligera con riesgo alto; revisar viento y oleaje antes de salir.")
        elif navigation_level == 1:
            parts.append("Navegación con precaución para embarcaciones pequeñas.")

    if surf_quality in ["very_good", "epic"]:
        parts.append("Condiciones favorables para surf en zonas adecuadas.")
    elif surf_quality in ["poor", "fair"]:
        parts.append("Calidad de surf limitada según el score físico previsto.")

    numeric = []

    if hs is not None:
        numeric.append(f"ola {hs:.2f} m")
    if period is not None:
        numeric.append(f"periodo {period:.1f} s")
    if wind_speed is not None:
        numeric.append(f"viento {wind_speed:.1f} m/s")

    if numeric:
        parts.append("Resumen físico: " + ", ".join(numeric) + ".")

    parts.append("DeepWave Canarias es una herramienta complementaria y no sustituye avisos oficiales.")

    return " ".join(parts)

## Celda 4 — Cargar predicciones físicas y resultados finales

In [5]:
if not PHYSICAL_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el parquet de predicciones físicas.\n"
        "Ejecuta primero 15_model_training_multitarget_physical.ipynb.\n"
        f"Ruta esperada: {PHYSICAL_PREDICTIONS_PATH}"
    )

physical_pred = pd.read_parquet(PHYSICAL_PREDICTIONS_PATH)
physical_pred["timestamp"] = normalize_timestamp(physical_pred["timestamp"])

# Normalizar columnas esperadas.
if "zona_id" not in physical_pred.columns:
    raise ValueError("El parquet de predicciones físicas no contiene columna zona_id.")

if "horizon_hours" not in physical_pred.columns:
    raise ValueError("El parquet de predicciones físicas no contiene columna horizon_hours.")

if "target_name" not in physical_pred.columns:
    raise ValueError("El parquet de predicciones físicas no contiene columna target_name.")

if "split" not in physical_pred.columns:
    physical_pred["split"] = "unknown"

if "isla" not in physical_pred.columns:
    physical_pred["isla"] = "UNKNOWN"

physical_pred["zona_id"] = physical_pred["zona_id"].astype(str)
physical_pred["isla"] = physical_pred["isla"].astype(str)
physical_pred["horizon_hours"] = pd.to_numeric(physical_pred["horizon_hours"], errors="coerce").astype("Int64")

value_col = choose_first_existing(
    physical_pred.columns,
    ["pred_lgbm", "prediction", "pred", "y_pred", "pred_value", "pred_degrees", "angular_pred"],
)

true_col = choose_first_existing(
    physical_pred.columns,
    ["y_true", "true", "target", "target_value", "observed"],
)

if value_col is None:
    raise ValueError(
        "No se encontró ninguna columna de predicción en predictions_val_test_core_targets.parquet. "
        "Candidatas: pred_lgbm, prediction, pred, y_pred, pred_value."
    )

# Filtrar targets prioritarios existentes.
available_targets = sorted(physical_pred["target_name"].dropna().astype(str).unique().tolist())
selected_targets = [t for t in TARGET_PRIORITY if t in available_targets]

if not selected_targets:
    raise ValueError(
        "No hay targets prioritarios disponibles en el parquet de predicciones. "
        f"Targets disponibles: {available_targets[:50]}"
    )

physical_pred = physical_pred[physical_pred["target_name"].isin(selected_targets)].copy()

print("Predicciones físicas cargadas:")
memory_report("physical_pred", physical_pred)
print("value_col:", value_col)
print("true_col:", true_col)
print("targets seleccionados:", selected_targets)
print("splits:", physical_pred["split"].value_counts(dropna=False).to_dict())
print("rango temporal:", physical_pred["timestamp"].min(), "→", physical_pred["timestamp"].max())

# Resultados finales opcionales para summaries.
final_architecture = read_json_safe(FINAL_RESULTS_DIR / "final_architecture_decision.json")
final_counts = read_json_safe(FINAL_RESULTS_DIR / "final_counts.json")

final_executive = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_executive_results_table.csv")
final_modeling_status = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_modeling_status.csv")
final_risk_recommendations = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_risk_recommendations.csv")
final_surf_recommendations = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_surf_recommendations.csv")

print("final_executive:", final_executive.shape)
print("final_modeling_status:", final_modeling_status.shape)

Predicciones físicas cargadas:
physical_pred: shape=(6755485, 15), memoria≈984.2 MB
value_col: pred_lgbm
true_col: y_true
targets seleccionados: ['hs', 'tp', 'wave_direction', 'wind_speed', 'wind_direction']
splits: {'test': 4667773, 'val': 2087712}
rango temporal: 2023-01-01 00:00:00+00:00 → 2025-12-31 20:00:00+00:00
final_executive: (65, 9)
final_modeling_status: (3, 6)


## Celda 5 — Crear catálogo de zonas

In [6]:
def load_zone_metadata_from_gold():
    """Carga solo columnas ligeras de zonas desde Gold multitarget si está disponible."""
    if not INPUT_GOLD_MULTITARGET_DIR.exists() or ds is None:
        return pd.DataFrame()

    try:
        dataset = ds.dataset(str(INPUT_GOLD_MULTITARGET_DIR), format="parquet", partitioning="hive")
        cols = dataset.schema.names

        candidate_cols = [
            "zona_id",
            "isla",
            "nombre_zona",
            "zone_name",
            "municipio",
            "lat",
            "lon",
            "latitude",
            "longitude",
            "latitud",
            "longitud",
            "coast_orientation_deg",
            "orientacion_costa",
            "coast_exposure_score",
            "bathymetry_depth_mean",
            "bathymetry_depth_min",
            "bathymetry_depth_max",
        ]

        selected = [c for c in candidate_cols if c in cols]

        if "zona_id" not in selected:
            return pd.DataFrame()

        table = dataset.to_table(columns=selected)
        meta = table.to_pandas()
        meta["zona_id"] = meta["zona_id"].astype(str)

        if "isla" not in meta.columns:
            meta["isla"] = "UNKNOWN"

        meta = meta.sort_values(["zona_id"]).drop_duplicates("zona_id").reset_index(drop=True)
        return meta

    except Exception as e:
        print("AVISO: no se pudo cargar metadatos de Gold:", e)
        return pd.DataFrame()


zone_meta = load_zone_metadata_from_gold()

if zone_meta.empty:
    print("No se pudo cargar zone_meta desde Gold. Se usará fallback desde predicciones.")
    zone_meta = (
        physical_pred[["zona_id", "isla"]]
        .drop_duplicates()
        .sort_values("zona_id")
        .reset_index(drop=True)
    )

# Candidatos de lat/lon.
lat_col = choose_first_existing(zone_meta.columns, ["lat", "latitude", "latitud"])
lon_col = choose_first_existing(zone_meta.columns, ["lon", "longitude", "longitud"])

# Nombre.
name_col = choose_first_existing(zone_meta.columns, ["nombre_zona", "zone_name", "name", "station_name"])

# Crear lat/lon fallback por isla para mapa si faltan.
island_centers = {
    "TENERIFE": (28.2916, -16.6291),
    "GRAN CANARIA": (27.9202, -15.5474),
    "LANZAROTE": (29.0469, -13.5899),
    "FUERTEVENTURA": (28.3587, -14.0537),
    "LA PALMA": (28.6819, -17.7642),
    "LA GOMERA": (28.1030, -17.2194),
    "EL HIERRO": (27.7255, -18.0243),
    "UNKNOWN": (28.3, -15.8),
}

def infer_island_center(isla):
    text = safe_str(isla, "UNKNOWN").upper()
    for key, coords in island_centers.items():
        if key in text:
            return coords
    return island_centers["UNKNOWN"]


zones = []

zone_meta = zone_meta.copy()

if MAX_ZONES is not None:
    zone_meta = zone_meta.head(MAX_ZONES).copy()

for idx, row in zone_meta.iterrows():
    zona_id = safe_str(row.get("zona_id"), f"zone_{idx}")
    isla = safe_str(row.get("isla"), "UNKNOWN")

    lat = safe_float(row.get(lat_col)) if lat_col else None
    lon = safe_float(row.get(lon_col)) if lon_col else None

    coords_source = "dataset"

    if lat is None or lon is None:
        center_lat, center_lon = infer_island_center(isla)
        # Offset pequeño para que los marcadores no se solapen si no hay coordenadas.
        offset = (idx % 7 - 3) * 0.035
        lat = round(center_lat + offset, 5)
        lon = round(center_lon + offset, 5)
        coords_source = "estimated_from_island_center"

    zone_name = safe_str(row.get(name_col), None) if name_col else None

    if not zone_name:
        zone_name = f"Zona {zona_id}"

    coast_orientation_deg = safe_float(row.get("coast_orientation_deg"))
    exposure = safe_float(row.get("coast_exposure_score"))

    zones.append({
        "zona_id": zona_id,
        "name": zone_name,
        "isla": isla,
        "latitude": lat,
        "longitude": lon,
        "coords_source": coords_source,
        "coast_orientation_deg": coast_orientation_deg,
        "coast_exposure_score": exposure,
        "metadata": {
            "municipio": safe_str(row.get("municipio"), None),
            "bathymetry_depth_mean": safe_float(row.get("bathymetry_depth_mean")),
            "bathymetry_depth_min": safe_float(row.get("bathymetry_depth_min")),
            "bathymetry_depth_max": safe_float(row.get("bathymetry_depth_max")),
        }
    })

zones_by_id = {z["zona_id"]: z for z in zones}

print("Zonas preparadas:", len(zones))
display(pd.DataFrame(zones).head(20))

Zonas preparadas: 14


,zona_id,name,isla,latitude,longitude,coords_source,coast_orientation_deg,coast_exposure_score,metadata
0,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,27.7837,-17.9045,dataset,270.0,0.4,"{'municipio': 'Valverde', 'bathymetry_depth_me..."
1,CAN_FV_GRAN_TARAJAL,Gran Tarajal,Fuerteventura,28.2113,-14.0195,dataset,45.0,0.8,"{'municipio': 'Tuineje', 'bathymetry_depth_mea..."
2,CAN_FV_PLAYA_DEL_VALLE,Playa del Valle,Fuerteventura,28.4854,-14.0943,dataset,45.0,0.8,"{'municipio': 'Betancuria', 'bathymetry_depth_..."
3,CAN_GC_SAN_CRISTOBAL,San Cristóbal,Gran Canaria,28.0771,-15.4145,dataset,90.0,0.4,"{'municipio': 'Las Palmas de Gran Canaria', 'b..."
4,CAN_LG_ERESES,Ereses,La Gomera,28.0246,-17.2350,dataset,270.0,0.4,"{'municipio': 'Alajeró', 'bathymetry_depth_mea..."
5,CAN_LP_CALLEJONCITOS,Callejoncitos,La Palma,28.8148,-17.9717,dataset,315.0,0.8,"{'municipio': 'Garafía', 'bathymetry_depth_mea..."
6,CAN_LP_EL_CHARCON,El Charcón,La Palma,28.6015,-17.9236,dataset,315.0,0.8,"{'municipio': 'Tazacorte', 'bathymetry_depth_m..."
7,CAN_LZ_ISLA_DE_LA_ALEGRANZA,Isla de la Alegranza,Alegranza,29.3858,-13.5094,dataset,45.0,0.8,"{'municipio': 'Teguise', 'bathymetry_depth_mea..."
8,CAN_LZ_PLAYA_DEL_COCHINO,Playa del Cochino,Lanzarote,29.0319,-13.8146,dataset,45.0,0.8,"{'municipio': 'Yaiza', 'bathymetry_depth_mean'..."
9,CAN_TF_AMARILLA,Amarilla,Tenerife,28.0092,-16.6385,dataset,270.0,0.4,"{'municipio': 'San Miguel de Abona', 'bathymet..."


## Celda 6 — Construir tabla wide de predicciones físicas

In [7]:
# Elegir split preferido disponible.
available_splits = physical_pred["split"].dropna().astype(str).unique().tolist()
selected_split = None

for split in PREFERRED_SPLIT_ORDER:
    if split in available_splits:
        selected_split = split
        break

if selected_split is None:
    selected_split = available_splits[0] if available_splits else "unknown"

physical_demo = physical_pred[physical_pred["split"].astype(str) == selected_split].copy()

if physical_demo.empty:
    physical_demo = physical_pred.copy()
    selected_split = "all"

print("Split seleccionado para app_data:", selected_split)
print("Filas demo:", len(physical_demo))

index_cols = ["zona_id", "timestamp", "split", "isla", "horizon_hours"]

physical_wide = (
    physical_demo
    .pivot_table(
        index=index_cols,
        columns="target_name",
        values=value_col,
        aggfunc="first",
    )
    .reset_index()
)

physical_wide.columns.name = None

# Añadir true values con prefijo true_ si están.
if true_col is not None:
    true_wide = (
        physical_demo
        .pivot_table(
            index=index_cols,
            columns="target_name",
            values=true_col,
            aggfunc="first",
        )
        .reset_index()
    )
    true_wide.columns.name = None

    rename_true = {c: f"true_{c}" for c in true_wide.columns if c not in index_cols}
    true_wide = true_wide.rename(columns=rename_true)

    physical_wide = physical_wide.merge(true_wide, on=index_cols, how="left")

# Filtrar a zonas catalogadas.
physical_wide["zona_id"] = physical_wide["zona_id"].astype(str)
physical_wide = physical_wide[physical_wide["zona_id"].isin(zones_by_id.keys())].copy()

# Horizonte como int.
physical_wide["horizon_hours"] = pd.to_numeric(physical_wide["horizon_hours"], errors="coerce").astype("Int64")
physical_wide = physical_wide[physical_wide["horizon_hours"].isin(HORIZONS_HOURS)].copy()

physical_wide = physical_wide.sort_values(["zona_id", "timestamp", "horizon_hours"]).reset_index(drop=True)

print("physical_wide:")
memory_report("physical_wide", physical_wide)
display(physical_wide.head(20))

Split seleccionado para app_data: test
Filas demo: 4667773
physical_wide:
physical_wide: shape=(979677, 11), memoria≈103.0 MB


,zona_id,timestamp,split,isla,horizon_hours,hs,tp,wind_speed,true_hs,true_tp,true_wind_speed
0,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,test,El Hierro,3,0.948810,9.175843,NaN,0.90,12.11,NaN
1,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,test,El Hierro,6,0.998246,9.832675,NaN,0.89,12.11,NaN
2,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,test,El Hierro,12,1.123022,7.684227,NaN,0.92,12.11,NaN
3,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,test,El Hierro,24,1.144070,7.026598,NaN,0.96,11.01,NaN
4,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,test,El Hierro,48,1.016312,7.081911,NaN,1.22,10.01,NaN
5,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,test,El Hierro,3,0.944457,10.667973,NaN,0.90,12.11,NaN
6,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,test,El Hierro,6,0.965401,10.545296,NaN,0.89,12.11,NaN
7,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,test,El Hierro,12,1.098766,9.468352,NaN,0.92,12.11,NaN
8,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,test,El Hierro,24,1.102685,8.136071,NaN,0.95,11.01,NaN
9,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,test,El Hierro,48,0.976253,8.195149,NaN,1.25,7.52,NaN


## Celda 7 — Seleccionar forecast demo por zona

In [8]:
def select_best_base_timestamp(group):
    """
    Selecciona el timestamp más reciente con mayor número de horizontes disponibles.
    """
    counts = (
        group
        .groupby("timestamp", as_index=False)
        .agg(
            n_horizons=("horizon_hours", "nunique"),
            has_hs=("hs", lambda s: int(s.notna().any()) if "hs" in group.columns else 0),
        )
    )

    if counts.empty:
        return None

    # Preferir timestamps con hs y más horizontes; en empate, el más reciente.
    counts = counts.sort_values(["has_hs", "n_horizons", "timestamp"], ascending=[False, False, False])
    return counts.iloc[0]["timestamp"]


forecast_rows = []

for zona_id, group in physical_wide.groupby("zona_id"):
    base_ts = select_best_base_timestamp(group)

    if base_ts is None:
        continue

    selected = group[group["timestamp"] == base_ts].copy()

    # Si faltan horizontes, completar con último registro disponible por horizonte.
    present_horizons = set(selected["horizon_hours"].dropna().astype(int).tolist())
    missing_horizons = [h for h in HORIZONS_HOURS if h not in present_horizons]

    if missing_horizons:
        fallback = (
            group[group["horizon_hours"].isin(missing_horizons)]
            .sort_values("timestamp")
            .groupby("horizon_hours", as_index=False)
            .tail(1)
        )
        selected = pd.concat([selected, fallback], ignore_index=True)

    selected = selected.sort_values("horizon_hours")
    forecast_rows.append(selected)

if forecast_rows:
    serving_wide = pd.concat(forecast_rows, ignore_index=True)
else:
    serving_wide = pd.DataFrame()

if serving_wide.empty:
    raise ValueError("No se pudieron construir forecasts por zona. Revisa physical_wide.")

serving_wide = serving_wide.sort_values(["zona_id", "horizon_hours"]).reset_index(drop=True)

print("serving_wide:")
memory_report("serving_wide", serving_wide)
display(serving_wide.head(30))

print("Horizontes por zona:")
display(serving_wide.groupby("zona_id")["horizon_hours"].nunique().describe())

serving_wide:
serving_wide: shape=(70, 11), memoria≈0.0 MB


,zona_id,timestamp,split,isla,horizon_hours,hs,tp,wind_speed,true_hs,true_tp,true_wind_speed
0,CAN_EH_PUERTO_DE_LA_ESTACA,2025-12-29 23:00:00+00:00,test,El Hierro,3,1.771585,15.696418,NaN,1.75,16.120001,NaN
1,CAN_EH_PUERTO_DE_LA_ESTACA,2025-12-29 23:00:00+00:00,test,El Hierro,6,1.892693,15.775272,NaN,1.85,16.120001,NaN
2,CAN_EH_PUERTO_DE_LA_ESTACA,2025-12-29 23:00:00+00:00,test,El Hierro,12,2.142047,15.097416,NaN,1.96,14.660000,NaN
3,CAN_EH_PUERTO_DE_LA_ESTACA,2025-12-29 23:00:00+00:00,test,El Hierro,24,2.013419,13.268606,NaN,1.92,13.320000,NaN
4,CAN_EH_PUERTO_DE_LA_ESTACA,2025-12-29 23:00:00+00:00,test,El Hierro,48,1.764580,12.384638,NaN,1.67,13.320000,NaN
5,CAN_FV_GRAN_TARAJAL,2025-12-29 23:00:00+00:00,test,Fuerteventura,3,0.188337,14.007721,NaN,0.16,17.740000,NaN
6,CAN_FV_GRAN_TARAJAL,2025-12-29 23:00:00+00:00,test,Fuerteventura,6,0.304282,11.478462,NaN,0.16,16.120001,NaN
7,CAN_FV_GRAN_TARAJAL,2025-12-29 23:00:00+00:00,test,Fuerteventura,12,0.490205,12.910935,NaN,0.17,16.120001,NaN
8,CAN_FV_GRAN_TARAJAL,2025-12-29 23:00:00+00:00,test,Fuerteventura,24,0.439571,10.187279,NaN,0.20,14.660000,NaN
9,CAN_FV_GRAN_TARAJAL,2025-12-29 23:00:00+00:00,test,Fuerteventura,48,0.661004,8.381348,NaN,0.18,13.320000,NaN


Horizontes por zona:


count    14.0
mean      5.0
std       0.0
min       5.0
25%       5.0
50%       5.0
75%       5.0
max       5.0
Name: horizon_hours, dtype: float64

## Celda 8 — Derivar riesgo, surf score y payloads finales

In [9]:
def get_value(row, candidates):
    for c in candidates:
        if c in row.index:
            v = safe_float(row.get(c))
            if v is not None:
                return v
    return None


flat_predictions = []
forecast_by_zone = {}

for zona_id, group in serving_wide.groupby("zona_id"):
    zone = zones_by_id.get(zona_id, {
        "zona_id": zona_id,
        "name": f"Zona {zona_id}",
        "isla": safe_str(group["isla"].iloc[0], "UNKNOWN") if "isla" in group.columns else "UNKNOWN",
        "latitude": None,
        "longitude": None,
        "coast_orientation_deg": None,
        "coast_exposure_score": None,
    })

    zone_forecasts = []

    for _, row in group.sort_values("horizon_hours").iterrows():
        horizon = safe_int(row.get("horizon_hours"))
        base_time = row.get("timestamp")
        valid_time = None

        if isinstance(base_time, pd.Timestamp) and horizon is not None:
            valid_time = base_time + pd.Timedelta(hours=horizon)

        hs = get_value(row, ["hs"])
        tm02 = get_value(row, ["tm02"])
        tp = get_value(row, ["tp"])
        period = tm02 if tm02 is not None else tp

        swell_height = get_value(row, ["swell_height"])
        swell_period = get_value(row, ["swell_period"])
        wave_direction = get_value(row, ["wave_direction"])
        swell_direction = get_value(row, ["swell_direction"])
        wind_speed = get_value(row, ["wind_speed"])
        wind_direction = get_value(row, ["wind_direction"])
        sea_level = get_value(row, ["sea_level"])
        daily_tidal_range = get_value(row, ["daily_tidal_range"])
        wind_wave_height = get_value(row, ["wind_wave_height"])

        coast_orientation_deg = zone.get("coast_orientation_deg")
        exposure = zone.get("coast_exposure_score")

        risk_general_level = assign_risk_general(hs)
        risk_beach_level = assign_risk_beach(hs, period=period, wind_speed=wind_speed, exposure=exposure)
        risk_navigation_level = assign_risk_navigation(hs, period=period, wind_speed=wind_speed)

        surf_score = surf_score_from_conditions(
            hs=hs,
            period=period,
            wind_speed=wind_speed,
            wave_direction=wave_direction,
            wind_direction=wind_direction,
            coast_orientation_deg=coast_orientation_deg,
        )

        payload = {
            "zona_id": zona_id,
            "zone_name": zone.get("name"),
            "isla": zone.get("isla"),
            "base_time": base_time.isoformat() if isinstance(base_time, pd.Timestamp) else safe_str(base_time),
            "valid_time": valid_time.isoformat() if isinstance(valid_time, pd.Timestamp) else None,
            "horizon_hours": horizon,
            "physical": {
                "hs": hs,
                "tm02": tm02,
                "tp": tp,
                "period": period,
                "swell_height": swell_height,
                "swell_period": swell_period,
                "wave_direction": wave_direction,
                "swell_direction": swell_direction,
                "wind_speed": wind_speed,
                "wind_direction": wind_direction,
                "sea_level": sea_level,
                "daily_tidal_range": daily_tidal_range,
                "wind_wave_height": wind_wave_height,
            },
            "risk_general": risk_payload(risk_general_level),
            "risk_beach": risk_payload(risk_beach_level),
            "risk_navigation": risk_payload(risk_navigation_level),
            "surf": surf_payload(surf_score),
            "quality": {
                "source": "precomputed_model_predictions",
                "split": selected_split,
                "is_demo_forecast": True,
            },
        }

        # Flatten helper fields for frontend.
        payload["hs"] = hs
        payload["period"] = period
        payload["wind_speed"] = wind_speed

        payload["recommendation"] = recommendation_text(payload)

        zone_forecasts.append(payload)
        flat_predictions.append(payload)

    forecast_by_zone[zona_id] = {
        "zone": zone,
        "forecast": zone_forecasts,
        "available_horizons": [p["horizon_hours"] for p in zone_forecasts],
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }

latest_predictions = []

for zona_id, payload in forecast_by_zone.items():
    forecasts = payload["forecast"]

    # Para mapa/dashboard, usar +24h si existe; si no, el primer horizonte disponible.
    chosen = None
    for h in [24, 12, 6, 3, 48]:
        matches = [r for r in forecasts if r["horizon_hours"] == h]
        if matches:
            chosen = matches[0]
            break

    if chosen is None and forecasts:
        chosen = forecasts[0]

    if chosen is not None:
        latest_predictions.append({
            "zone": payload["zone"],
            "selected_forecast": chosen,
        })

print("Zonas con forecast:", len(forecast_by_zone))
print("Predicciones planas:", len(flat_predictions))
print("Latest predictions:", len(latest_predictions))

display(pd.DataFrame([
    {
        "zona_id": p["zona_id"],
        "horizon": p["horizon_hours"],
        "hs": p["physical"]["hs"],
        "period": p["physical"]["period"],
        "wind_speed": p["physical"]["wind_speed"],
        "risk_general": p["risk_general"]["label_es"],
        "risk_beach": p["risk_beach"]["label_es"],
        "risk_navigation": p["risk_navigation"]["label_es"],
        "surf_score": p["surf"]["score"],
        "surf_quality": p["surf"]["quality"],
    }
    for p in flat_predictions[:30]
]))

Zonas con forecast: 14
Predicciones planas: 70
Latest predictions: 14


,zona_id,horizon,hs,period,wind_speed,risk_general,risk_beach,risk_navigation,surf_score,surf_quality
0,CAN_EH_PUERTO_DE_LA_ESTACA,3,1.7716,15.6964,NaN,medio,medio,alto,8.0,epic
1,CAN_EH_PUERTO_DE_LA_ESTACA,6,1.8927,15.7753,NaN,medio,alto,alto,8.0,epic
2,CAN_EH_PUERTO_DE_LA_ESTACA,12,2.1420,15.0974,NaN,alto,alto,extremo,8.0,epic
3,CAN_EH_PUERTO_DE_LA_ESTACA,24,2.0134,13.2686,NaN,alto,alto,extremo,8.0,epic
4,CAN_EH_PUERTO_DE_LA_ESTACA,48,1.7646,12.3846,NaN,medio,medio,alto,8.0,epic
5,CAN_FV_GRAN_TARAJAL,3,0.1883,14.0077,NaN,bajo,bajo,bajo,2.0,fair
6,CAN_FV_GRAN_TARAJAL,6,0.3043,11.4785,NaN,bajo,bajo,bajo,1.0,poor
7,CAN_FV_GRAN_TARAJAL,12,0.4902,12.9109,NaN,bajo,bajo,bajo,4.0,good
8,CAN_FV_GRAN_TARAJAL,24,0.4396,10.1873,NaN,bajo,bajo,bajo,3.0,fair
9,CAN_FV_GRAN_TARAJAL,48,0.6610,8.3813,NaN,bajo,bajo,bajo,2.0,fair


## Celda 9 — Crear ejemplos de demo para API y presentación

In [10]:
def risk_rank(record):
    levels = [
        record.get("risk_general", {}).get("level"),
        record.get("risk_beach", {}).get("level"),
        record.get("risk_navigation", {}).get("level"),
    ]
    levels = [l for l in levels if l is not None]
    return max(levels) if levels else -1


def find_best_record(records, key_fn, reverse=True):
    valid = []
    for r in records:
        try:
            score = key_fn(r)
            if score is None or pd.isna(score):
                continue
            valid.append((score, r))
        except Exception:
            pass

    if not valid:
        return None

    valid = sorted(valid, key=lambda x: x[0], reverse=reverse)
    return valid[0][1]


highest_risk = find_best_record(flat_predictions, risk_rank, reverse=True)
lowest_risk = find_best_record(flat_predictions, risk_rank, reverse=False)
best_surf = find_best_record(flat_predictions, lambda r: r.get("surf", {}).get("score"), reverse=True)
worst_surf = find_best_record(flat_predictions, lambda r: r.get("surf", {}).get("score"), reverse=False)
navigation_warning = find_best_record(flat_predictions, lambda r: r.get("risk_navigation", {}).get("level"), reverse=True)

demo_examples = {
    "highest_risk": highest_risk,
    "lowest_risk": lowest_risk,
    "best_surf": best_surf,
    "worst_surf": worst_surf,
    "navigation_warning": navigation_warning,
    "curl_examples": [
        "curl http://127.0.0.1:8000/health",
        "curl http://127.0.0.1:8000/zones",
        "curl http://127.0.0.1:8000/predict/{zona_id}",
        "curl http://127.0.0.1:8000/predict/{zona_id}?horizon=24",
        "curl http://127.0.0.1:8000/predict/all?horizon=24",
        "curl http://127.0.0.1:8000/model/summary",
    ],
}

print("Ejemplos demo creados:")
for k, v in demo_examples.items():
    if isinstance(v, dict):
        print(k, "->", v.get("zona_id"), "+", v.get("horizon_hours"), "h")

Ejemplos demo creados:
highest_risk -> CAN_EH_PUERTO_DE_LA_ESTACA + 12 h
lowest_risk -> CAN_FV_GRAN_TARAJAL + 3 h
best_surf -> CAN_FV_PLAYA_DEL_VALLE + 3 h
worst_surf -> CAN_FV_GRAN_TARAJAL + 6 h
navigation_warning -> CAN_EH_PUERTO_DE_LA_ESTACA + 12 h


## Celda 10 — Crear resumen de modelo, contrato API y leyendas

In [11]:
# Métricas destacadas.
def metric_from_executive(module, horizon, metric_col="metric_value"):
    if final_executive.empty:
        return None

    sub = final_executive.copy()

    if "module" in sub.columns:
        sub = sub[sub["module"] == module]

    if "horizon_hours" in sub.columns:
        sub = sub[pd.to_numeric(sub["horizon_hours"], errors="coerce") == horizon]

    if sub.empty or metric_col not in sub.columns:
        return None

    return safe_float(sub.iloc[0][metric_col])


model_summary = {
    "project": "DeepWave Canarias",
    "version": "production_artifacts_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_split": selected_split,
    "description": "Artefactos ligeros para servir predicciones marítimas, riesgo y surf score desde FastAPI y dashboard web.",
    "data_sources": {
        "physical_predictions": str(PHYSICAL_PREDICTIONS_PATH.relative_to(BASE_DIR)) if PHYSICAL_PREDICTIONS_PATH.exists() else None,
        "final_results": str(FINAL_RESULTS_DIR.relative_to(BASE_DIR)) if FINAL_RESULTS_DIR.exists() else None,
        "gold_multitarget": str(INPUT_GOLD_MULTITARGET_DIR.relative_to(BASE_DIR)) if INPUT_GOLD_MULTITARGET_DIR.exists() else None,
    },
    "modeling_blocks": {
        "physical": "LightGBMRegressor por variable y horizonte",
        "risk": "PhysicalDerivedRisk y PersistenceRisk según horizonte/módulo",
        "surf": "PhysicalDerivedSurfScore como arquitectura final",
    },
    "horizons_hours": HORIZONS_HOURS,
    "physical_targets_included": selected_targets,
    "zones_count": len(zones),
    "predictions_count": len(flat_predictions),
    "metrics_highlights": {
        "hs_mae_24h": metric_from_executive("hs", 24),
        "hs_mae_48h": metric_from_executive("hs", 48),
        "risk_general_score_24h": metric_from_executive("risk_general", 24),
        "surf_score_mae_24h": metric_from_executive("surf_score", 24),
    },
    "important_note": "Predicciones de demo precalculadas desde el conjunto test/validación. Para producción real se conectaría la API al pipeline de inferencia actualizado.",
}

api_contract = {
    "service": "DeepWave Canarias API",
    "version": "1.0.0",
    "base_url_local": "http://127.0.0.1:8000",
    "endpoints": [
        {
            "method": "GET",
            "path": "/health",
            "description": "Comprueba que la API está activa.",
            "response_file": None,
        },
        {
            "method": "GET",
            "path": "/zones",
            "description": "Lista zonas disponibles con coordenadas y metadatos.",
            "response_file": "zones.json",
        },
        {
            "method": "GET",
            "path": "/predict/{zona_id}",
            "description": "Devuelve forecast completo por zona.",
            "response_file": "forecast_by_zone.json",
            "params": [{"name": "horizon", "type": "int", "required": False, "values": HORIZONS_HOURS}],
        },
        {
            "method": "GET",
            "path": "/predict/all",
            "description": "Devuelve predicción seleccionada para todas las zonas. Útil para mapa.",
            "response_file": "latest_predictions.json",
            "params": [{"name": "horizon", "type": "int", "required": False, "values": HORIZONS_HOURS}],
        },
        {
            "method": "GET",
            "path": "/risk/{zona_id}",
            "description": "Devuelve riesgos general, playa y navegación por zona.",
            "response_file": "forecast_by_zone.json",
        },
        {
            "method": "GET",
            "path": "/surf/{zona_id}",
            "description": "Devuelve surf score y categoría por zona.",
            "response_file": "forecast_by_zone.json",
        },
        {
            "method": "GET",
            "path": "/model/summary",
            "description": "Resumen del sistema de modelos y métricas.",
            "response_file": "model_summary.json",
        },
    ],
    "schemas": {
        "ForecastRecord": {
            "zona_id": "string",
            "zone_name": "string",
            "isla": "string",
            "base_time": "datetime",
            "valid_time": "datetime",
            "horizon_hours": "int",
            "physical": "object",
            "risk_general": "RiskPayload",
            "risk_beach": "RiskPayload",
            "risk_navigation": "RiskPayload",
            "surf": "SurfPayload",
            "recommendation": "string",
        },
        "RiskPayload": {
            "level": "int|null",
            "label": "string",
            "label_es": "string",
            "color": "hex",
        },
        "SurfPayload": {
            "score": "float|null",
            "quality": "string",
            "quality_es": "string",
            "color": "hex",
        },
    },
}

risk_legend = {
    "levels": [
        {"level": k, **v}
        for k, v in RISK_LABELS.items()
    ],
    "modules": ["general", "beach", "navigation"],
    "thresholds": RISK_THRESHOLDS,
}

surf_legend = {
    "quality_levels": [
        {"quality": k, **v}
        for k, v in SURF_LABELS.items()
    ],
    "score_config": SURF_SCORE_CONFIG,
}

frontend_config = {
    "api_base_url": "http://127.0.0.1:8000",
    "default_horizon": 24,
    "available_horizons": HORIZONS_HOURS,
    "map_center": {"lat": 28.3, "lon": -15.8, "zoom": 7},
    "risk_colors": {str(k): v["color"] for k, v in RISK_LABELS.items()},
    "surf_colors": {k: v["color"] for k, v in SURF_LABELS.items()},
    "default_layer": "risk_beach",
}

print("model_summary listo.")
print("api_contract endpoints:", len(api_contract["endpoints"]))

model_summary listo.
api_contract endpoints: 7


## Celda 11 — Escribir `app_data/`

In [12]:
write_json(APP_DATA_DIR / "zones.json", {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "count": len(zones),
    "zones": zones,
})

write_json(APP_DATA_DIR / "forecast_by_zone.json", {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_split": selected_split,
    "count": len(forecast_by_zone),
    "data": forecast_by_zone,
})

write_json(APP_DATA_DIR / "predictions_flat.json", {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_split": selected_split,
    "count": len(flat_predictions),
    "predictions": flat_predictions,
})

write_json(APP_DATA_DIR / "latest_predictions.json", {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_split": selected_split,
    "count": len(latest_predictions),
    "predictions": latest_predictions,
})

write_json(APP_DATA_DIR / "model_summary.json", model_summary)
write_json(APP_DATA_DIR / "api_contract.json", api_contract)
write_json(APP_DATA_DIR / "risk_legend.json", risk_legend)
write_json(APP_DATA_DIR / "surf_legend.json", surf_legend)
write_json(APP_DATA_DIR / "frontend_config.json", frontend_config)
write_json(APP_DATA_DIR / "demo_examples.json", demo_examples)

print("Archivos JSON principales escritos.")

JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/zones.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/forecast_by_zone.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/predictions_flat.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/latest_predictions.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/model_summary.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/api_contract.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/risk_legend.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/surf_legend.json
JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/fron

## Celda 12 — Crear informe de preparación para producción

In [13]:
report_lines = []

report_lines.append("# DeepWave Canarias — Production Serving Artifacts\n")
report_lines.append("## Objetivo\n")
report_lines.append(
    "Este paquete contiene artefactos ligeros para servir predicciones marítimas, riesgo y surf score "
    "desde una API FastAPI y una aplicación cliente web."
)

report_lines.append("\n## Entradas utilizadas\n")
report_lines.append(f"- Predicciones físicas: `{PHYSICAL_PREDICTIONS_PATH.relative_to(BASE_DIR)}`")
report_lines.append(f"- Resultados finales: `{FINAL_RESULTS_DIR.relative_to(BASE_DIR) if FINAL_RESULTS_DIR.exists() else 'no disponible'}`")
report_lines.append(f"- Gold multitarget: `{INPUT_GOLD_MULTITARGET_DIR.relative_to(BASE_DIR) if INPUT_GOLD_MULTITARGET_DIR.exists() else 'no disponible'}`")

report_lines.append("\n## Salidas generadas\n")
for path_str in written_files:
    p = Path(path_str)
    try:
        rel = p.relative_to(BASE_DIR)
    except Exception:
        rel = p
    report_lines.append(f"- `{rel}`")

report_lines.append("\n## Contenido del paquete\n")
report_lines.append(f"- Zonas disponibles: **{len(zones)}**")
report_lines.append(f"- Predicciones planas: **{len(flat_predictions)}**")
report_lines.append(f"- Zonas con forecast: **{len(forecast_by_zone)}**")
report_lines.append(f"- Horizontes: **{HORIZONS_HOURS}**")
report_lines.append(f"- Split usado para demo: **{selected_split}**")
report_lines.append(f"- Targets físicos incluidos: **{', '.join(selected_targets)}**")

report_lines.append("\n## Endpoints recomendados para FastAPI\n")
for ep in api_contract["endpoints"]:
    report_lines.append(f"- `{ep['method']} {ep['path']}` — {ep['description']}")

report_lines.append("\n## Estrategia de producción\n")
report_lines.append(
    "La primera versión de producción usará predicciones precalculadas. "
    "Esto reduce consumo de RAM, evita cargar modelos pesados en la API y permite una demo estable."
)

report_lines.append("\n## Próximo paso\n")
report_lines.append(
    "Crear el backend FastAPI leyendo los JSON de `app_data/` y exponer los endpoints definidos en `api_contract.json`."
)

report_lines.append("\n## Nota importante\n")
report_lines.append(
    "Estos datos son artefactos de demostración basados en predicciones del conjunto de validación/test. "
    "Para una producción real operativa, el pipeline debería actualizar `app_data/` con predicciones recientes."
)

production_report = "\n".join(report_lines)

write_text(APP_DATA_DIR / "production_readiness_report.md", production_report)

# Copia a docs para memoria si existe.
write_text(DOCS_DIR / "production_serving_artifacts.md", production_report)

print(production_report[:4000])

Texto escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/production_readiness_report.md
Texto escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/production_serving_artifacts.md
# DeepWave Canarias — Production Serving Artifacts

## Objetivo

Este paquete contiene artefactos ligeros para servir predicciones marítimas, riesgo y surf score desde una API FastAPI y una aplicación cliente web.

## Entradas utilizadas

- Predicciones físicas: `gold/model_results/multitarget_physical/predictions_val_test_core_targets.parquet`
- Resultados finales: `gold/model_results/final_report_multitarget`
- Gold multitarget: `gold/multitarget_training_dataset`

## Salidas generadas

- `app_data/zones.json`
- `app_data/forecast_by_zone.json`
- `app_data/predictions_flat.json`
- `app_data/latest_predictions.json`
- `app_data/model_summary.json`
- `app_data/api_contract.json`
- `app_data/risk_legend.json`
- `app_data/surf_legend.json`
- `

## Celda 13 — Crear manifest de producción

In [14]:
manifest = []

for p in sorted(APP_DATA_DIR.glob("*")):
    if p.is_file():
        manifest.append({
            "file": p.name,
            "relative_path": str(p.relative_to(BASE_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 2),
            "modified": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
        })

production_manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "base_dir": str(BASE_DIR),
    "app_data_dir": str(APP_DATA_DIR.relative_to(BASE_DIR)),
    "files": manifest,
    "quality_checks": {
        "zones_count": len(zones),
        "forecast_zones_count": len(forecast_by_zone),
        "flat_predictions_count": len(flat_predictions),
        "latest_predictions_count": len(latest_predictions),
        "selected_split": selected_split,
        "selected_targets": selected_targets,
    },
}

write_json(APP_DATA_DIR / "production_manifest.json", production_manifest)

manifest_df = pd.DataFrame(manifest)
display(manifest_df)

JSON escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/app_data/production_manifest.json


,file,relative_path,size_kb,modified
0,api_contract.json,app_data/api_contract.json,2.51,2026-05-21T11:53:24
1,demo_examples.json,app_data/demo_examples.json,8.17,2026-05-21T11:53:24
2,forecast_by_zone.json,app_data/forecast_by_zone.json,144.11,2026-05-21T11:53:24
3,frontend_config.json,app_data/frontend_config.json,0.48,2026-05-21T11:53:24
4,latest_predictions.json,app_data/latest_predictions.json,32.63,2026-05-21T11:53:24
5,model_summary.json,app_data/model_summary.json,1.31,2026-05-21T11:53:24
6,predictions_flat.json,app_data/predictions_flat.json,119.10,2026-05-21T11:53:24
7,production_readiness_report.md,app_data/production_readiness_report.md,2.16,2026-05-21T11:53:24
8,risk_legend.json,app_data/risk_legend.json,0.94,2026-05-21T11:53:24
9,surf_legend.json,app_data/surf_legend.json,0.83,2026-05-21T11:53:24


## Celda 14 — Validaciones de producción

In [15]:
required_files = [
    APP_DATA_DIR / "zones.json",
    APP_DATA_DIR / "forecast_by_zone.json",
    APP_DATA_DIR / "predictions_flat.json",
    APP_DATA_DIR / "latest_predictions.json",
    APP_DATA_DIR / "model_summary.json",
    APP_DATA_DIR / "api_contract.json",
    APP_DATA_DIR / "risk_legend.json",
    APP_DATA_DIR / "surf_legend.json",
    APP_DATA_DIR / "frontend_config.json",
    APP_DATA_DIR / "demo_examples.json",
    APP_DATA_DIR / "production_readiness_report.md",
    APP_DATA_DIR / "production_manifest.json",
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError("Faltan archivos obligatorios en app_data: " + json.dumps(missing, indent=2, ensure_ascii=False))

if len(zones) == 0:
    raise ValueError("zones está vacío.")

if len(forecast_by_zone) == 0:
    raise ValueError("forecast_by_zone está vacío.")

if len(flat_predictions) == 0:
    raise ValueError("predictions_flat está vacío.")

if len(latest_predictions) == 0:
    raise ValueError("latest_predictions está vacío.")

# Validar que al menos hay hs o viento en algún registro.
has_hs = any(p.get("physical", {}).get("hs") is not None for p in flat_predictions)
has_risk = any(p.get("risk_general", {}).get("level") is not None for p in flat_predictions)
has_surf = any(p.get("surf", {}).get("score") is not None for p in flat_predictions)

if not has_hs:
    raise ValueError("No hay hs en las predicciones planas.")

if not has_risk:
    raise ValueError("No se pudo calcular riesgo general en ningún registro.")

if not has_surf:
    raise ValueError("No se pudo calcular surf score en ningún registro.")

# Validar que los JSON son legibles.
for p in required_files:
    if p.suffix == ".json":
        _ = read_json_safe(p)

print("Validación de producción OK.")
print("\nArchivos app_data:")
for p in required_files:
    size_kb = round(p.stat().st_size / 1024, 2)
    print(f"- {p.name}: {size_kb} KB")

print("\nPrimeras zonas:")
display(pd.DataFrame(zones).head(10))

print("\nPrimeras predicciones planas:")
display(pd.DataFrame([
    {
        "zona_id": p["zona_id"],
        "zone_name": p["zone_name"],
        "horizon": p["horizon_hours"],
        "hs": p["physical"]["hs"],
        "period": p["physical"]["period"],
        "wind_speed": p["physical"]["wind_speed"],
        "risk_beach": p["risk_beach"]["label_es"],
        "risk_navigation": p["risk_navigation"]["label_es"],
        "surf_score": p["surf"]["score"],
        "surf_quality": p["surf"]["quality_es"],
    }
    for p in flat_predictions[:20]
]))

print("\n✅ Artefactos de producción generados correctamente.")

Validación de producción OK.

Archivos app_data:
- zones.json: 6.29 KB
- forecast_by_zone.json: 144.11 KB
- predictions_flat.json: 119.1 KB
- latest_predictions.json: 32.63 KB
- model_summary.json: 1.31 KB
- api_contract.json: 2.51 KB
- risk_legend.json: 0.94 KB
- surf_legend.json: 0.83 KB
- frontend_config.json: 0.48 KB
- demo_examples.json: 8.17 KB
- production_readiness_report.md: 2.16 KB
- production_manifest.json: 2.29 KB

Primeras zonas:


,zona_id,name,isla,latitude,longitude,coords_source,coast_orientation_deg,coast_exposure_score,metadata
0,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,27.7837,-17.9045,dataset,270.0,0.4,"{'municipio': 'Valverde', 'bathymetry_depth_me..."
1,CAN_FV_GRAN_TARAJAL,Gran Tarajal,Fuerteventura,28.2113,-14.0195,dataset,45.0,0.8,"{'municipio': 'Tuineje', 'bathymetry_depth_mea..."
2,CAN_FV_PLAYA_DEL_VALLE,Playa del Valle,Fuerteventura,28.4854,-14.0943,dataset,45.0,0.8,"{'municipio': 'Betancuria', 'bathymetry_depth_..."
3,CAN_GC_SAN_CRISTOBAL,San Cristóbal,Gran Canaria,28.0771,-15.4145,dataset,90.0,0.4,"{'municipio': 'Las Palmas de Gran Canaria', 'b..."
4,CAN_LG_ERESES,Ereses,La Gomera,28.0246,-17.2350,dataset,270.0,0.4,"{'municipio': 'Alajeró', 'bathymetry_depth_mea..."
5,CAN_LP_CALLEJONCITOS,Callejoncitos,La Palma,28.8148,-17.9717,dataset,315.0,0.8,"{'municipio': 'Garafía', 'bathymetry_depth_mea..."
6,CAN_LP_EL_CHARCON,El Charcón,La Palma,28.6015,-17.9236,dataset,315.0,0.8,"{'municipio': 'Tazacorte', 'bathymetry_depth_m..."
7,CAN_LZ_ISLA_DE_LA_ALEGRANZA,Isla de la Alegranza,Alegranza,29.3858,-13.5094,dataset,45.0,0.8,"{'municipio': 'Teguise', 'bathymetry_depth_mea..."
8,CAN_LZ_PLAYA_DEL_COCHINO,Playa del Cochino,Lanzarote,29.0319,-13.8146,dataset,45.0,0.8,"{'municipio': 'Yaiza', 'bathymetry_depth_mean'..."
9,CAN_TF_AMARILLA,Amarilla,Tenerife,28.0092,-16.6385,dataset,270.0,0.4,"{'municipio': 'San Miguel de Abona', 'bathymet..."



Primeras predicciones planas:


,zona_id,zone_name,horizon,hs,period,wind_speed,risk_beach,risk_navigation,surf_score,surf_quality
0,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,3,1.7716,15.6964,NaN,medio,alto,8.0,excelente
1,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,6,1.8927,15.7753,NaN,alto,alto,8.0,excelente
2,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,12,2.1420,15.0974,NaN,alto,extremo,8.0,excelente
3,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,24,2.0134,13.2686,NaN,alto,extremo,8.0,excelente
4,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,48,1.7646,12.3846,NaN,medio,alto,8.0,excelente
5,CAN_FV_GRAN_TARAJAL,Gran Tarajal,3,0.1883,14.0077,NaN,bajo,bajo,2.0,regular
6,CAN_FV_GRAN_TARAJAL,Gran Tarajal,6,0.3043,11.4785,NaN,bajo,bajo,1.0,malo
7,CAN_FV_GRAN_TARAJAL,Gran Tarajal,12,0.4902,12.9109,NaN,bajo,bajo,4.0,bueno
8,CAN_FV_GRAN_TARAJAL,Gran Tarajal,24,0.4396,10.1873,NaN,bajo,bajo,3.0,regular
9,CAN_FV_GRAN_TARAJAL,Gran Tarajal,48,0.6610,8.3813,NaN,bajo,bajo,2.0,regular



✅ Artefactos de producción generados correctamente.


## Resultado esperado

Al final debe aparecer:

```text
✅ Artefactos de producción generados correctamente.
```

Después de este notebook, la siguiente fase será crear el backend:

```text
src/api/main.py
src/api/services.py
src/api/schemas.py
```

La API deberá leer:

```text
app_data/*.json
```

y exponer endpoints como:

```text
GET /health
GET /zones
GET /predict/{zona_id}
GET /predict/all
GET /risk/{zona_id}
GET /surf/{zona_id}
GET /model/summary
```

Luego se podrá crear el cliente web usando:

```text
frontend/index.html
frontend/styles.css
frontend/app.js
```